# Cross-encoder only (Kaggle) — robust ~30-min dictionary lever

Kaggle version of `cross_encoder_only.ipynb`. Free T4 GPU on a separate quota
from Colab.

**Before running, in the right-hand panel:**
1. *Session options → Accelerator → **GPU T4 x2***
2. *Session options → Internet → **On*** (required for git clone / pip / wget)

Outputs land in `/kaggle/working/` — download `cross_encoder_outputs.tar.gz`
from the **Output** tab when done.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone + deps

In [ ]:
%cd /kaggle/working
!rm -rf embedding-aligner
!git clone --depth=1 https://github.com/komapc/embedding-aligner.git
%cd embedding-aligner
!pip install -q -r requirements.txt

## 3. Fetch the cross-encoder positives

In [ ]:
REL = 'https://github.com/komapc/embedding-aligner/releases/download/cross-encoder-inputs'
!wget -q {REL}/cross_encoder_inputs.tar.gz
!mkdir -p extractor_work && tar xzf cross_encoder_inputs.tar.gz -C extractor_work --strip-components=1
!ls -lh extractor_work/ results/bert_ido_epo_alignment/translation_candidates.json

## 4. Train the cross-encoder  — ~25 min

**Note the final held-out F1 / AUC / precision.**

In [ ]:
!python3 scripts/16_train_cross_encoder.py \
  --bilingual-raw extractor_work/bilingual_raw.json \
  --langlinks extractor_work/io_eo_langlinks.json \
  --candidates results/bert_ido_epo_alignment/translation_candidates.json \
  --eo-vocab data/esperanto_vocabulary.txt \
  --model-out models/cross-encoder-io-eo --epochs 3 --batch-size 32

## 5. Apply the cross-encoder → re-ranked pairs

In [ ]:
!python3 scripts/17_apply_cross_encoder.py \
  --model models/cross-encoder-io-eo \
  --candidates results/bert_ido_epo_alignment/translation_candidates.json \
  --output results/bert_ido_epo_alignment/translation_candidates_ce.json \
  --threshold 0.5 --top-k 3

## 6. Stage outputs in /kaggle/working (download from the Output tab)

In [ ]:
!tar czf /kaggle/working/cross_encoder_outputs.tar.gz \
  results/bert_ido_epo_alignment/translation_candidates_ce.json \
  models/cross-encoder-io-eo
!ls -lh /kaggle/working/cross_encoder_outputs.tar.gz